# Tema Programare: Crearea si Ingineria Trasaturilor

Bun venit la tema de programare despre Crearea si Ingineria Trasaturilor!

Datele brute spun rareori intreaga poveste. Feature engineering - arta de a crea variabile noi, semnificative, pornind de la cele existente - este adesea cel mai important pas in construirea unor modele cu performanta ridicata. In aceasta tema vei construi trasaturi pe care niciun algoritm automat nu le-ar putea deduce singur: tipare temporale, semnale text si agregari bazate pe cunostinte de domeniu.

**Ce vei face in aceasta tema:**

* **Trasaturi RFM** - Construiesti trasaturi de tip Recency, Frequency si Monetary pornind de la timestamp-uri si sume ale tranzactiilor
* **Transformari matematice** - Creezi trasaturi de tip raport, valori transformate log si termeni de interactiune pereche
* **Trasaturi DateTime** - Extragi ora din zi, ziua saptamanii, luna si indicatori de weekend din coloane timestamp
* **Trasaturi text** - Parsezi string-uri de nume de produs in indicatori pentru cuvinte-cheie, numar de cuvinte si flag-uri de prezenta
* **Trasaturi polinomiale (sklearn)** - Folosesti `PolynomialFeatures` pentru a genera sistematic termeni de putere si de interactiune
* **Trasaturi polinomiale de la zero** - Implementezi generarea trasaturilor polinomiale folosind `itertools.combinations_with_replacement`

Sa incepem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI:</h4>

* Toate celulele sunt blocate, cu exceptia celor in care trebuie sa trimiti solutiile sau cand este mentionat explicit ca poti interactiona cu ele.

* In fiecare celula de exercitiu cauta comentariile `### START CODE HERE ###` si `### END CODE HERE ###`. Acestea iti arata unde sa scrii codul. **Nu adauga si nu modifica cod in afara acestor comentarii**.

* Poti adauga celule noi pentru experimente, dar acestea vor fi ignorate de evaluator, asa ca nu te baza pe celulele nou create pentru codul solutiei. Foloseste locurile oferite in notebook.

* Evita folosirea variabilelor globale daca nu este absolut necesar. Evaluatorul iti testeaza codul intr-un mediu izolat, fara sa ruleze toate celulele de sus in jos. Din acest motiv, variabilele globale pot sa nu fie disponibile la evaluare. Variabilele globale care trebuie folosite vor fi definite cu MAJUSCULE.

* Pentru a trimite notebook-ul la evaluare, salveaza-l mai intai apasand pe iconita 💾 din coltul stanga sus al paginii, apoi apasa pe butonul `Submit assignment` din coltul dreapta sus.
---


## Cuprins

- [Exercitiul 1 - Trasaturi de domeniu (RFM)](#exercise-1)
- [Exercitiul 2 - Trasaturi matematice](#exercise-2)
- [Exercitiul 3 - Trasaturi DateTime](#exercise-3)
- [Exercitiul 4 - Trasaturi text](#exercise-4)
- [Exercitiul 5 - Trasaturi polinomiale (sklearn)](#exercise-5)
- [Exercitiul 6 - Trasaturi polinomiale de la zero](#exercise-6)


## Importuri si configurare


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests
from itertools import combinations_with_replacement
from sklearn.preprocessing import PolynomialFeatures
from sklearn.feature_extraction.text import TfidfVectorizer

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## Set de date

Vom folosi un set de date sintetic cu tranzactii ale clientilor, cu 500 de clienti. Fiecare rand reprezinta un client cu:

| Coloana | Descriere |
|:-------|:------------|
| `customer_id` | Identificator unic al clientului |
| `last_purchase_date` | Data celei mai recente achizitii |
| `purchase_count` | Numarul total de achizitii efectuate |
| `total_spent` | Suma totala cheltuita (in unitati monetare) |
| `n_products` | Numarul de tipuri distincte de produse cumparate |
| `product_name` | Numele celui mai recent produs cumparat |
| `signup_date` | Data la care clientul s-a inregistrat prima data |

Toate trasaturile bazate pe timp vor folosi `REFERENCE_DATE = pd.Timestamp('2024-01-01')` ca punct de referinta fix.


In [ ]:
np.random.seed(42)
n_customers = 500
REFERENCE_DATE = pd.Timestamp('2024-01-01')

last_purchase_dates = REFERENCE_DATE - pd.to_timedelta(
    np.random.randint(1, 365, n_customers), unit='D')
signup_dates = REFERENCE_DATE - pd.to_timedelta(
    np.random.randint(365, 1825, n_customers), unit='D')

df = pd.DataFrame({
    'customer_id': range(1, n_customers + 1),
    'last_purchase_date': last_purchase_dates,
    'purchase_count': np.random.randint(1, 50, n_customers),
    'total_spent': np.random.uniform(10, 5000, n_customers).round(2),
    'n_products': np.random.randint(1, 20, n_customers),
    'product_name': np.random.choice([
        'laptop computer', 'wireless headphones', 'coffee maker deluxe',
        'running shoes premium', 'smartphone case', 'gaming keyboard rgb'
    ], n_customers),
    'signup_date': signup_dates,
})
X_num = df[['purchase_count', 'total_spent', 'n_products']].values.astype(float)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst 3 rows:")
print(df.head(3).to_string())

Dataset shape: (500, 7)

First 3 rows:
   customer_id last_purchase_date  purchase_count  total_spent  n_products        product_name signup_date
0            1        2023-09-20              49      2844.05          15     smartphone case  2021-08-23
1            2        2023-01-17              12      4578.29           4  running shoes premium  2020-07-20
2            3        2023-04-05              47       179.39          11   gaming keyboard rgb  2022-07-26

<a id='exercise-1'></a>
## Exercitiul 1 - Trasaturi de domeniu (RFM)

### Context

Cadrul **RFM** surprinde comportamentul clientilor pe trei dimensiuni:
- **Recency** - cat de recent au cumparat? (mai mic = mai implicati)
- **Frequency** - cat de des cumpara? (mai mare = mai loiali)
- **Monetary** - cat cheltuie? (mai mare = mai valorosi)

Aceste trei metrici reprezinta baza segmentarii clientilor in retail, e-commerce si afaceri pe baza de abonament.

### Sarcina

Implementeaza `create_domain_features(df, reference_date)` care adauga **4 coloane noi** intr-o copie a lui `df`:

| Coloana noua | Formula |
|:-----------|:--------|
| `recency_days` | `(reference_date - last_purchase_date).days` |
| `avg_order_value` | `total_spent / purchase_count` |
| `customer_lifetime_days` | `(reference_date - signup_date).days` |
| `purchase_frequency` | `purchase_count / customer_lifetime_days` |

**Indicii:**
- Foloseste `.dt.days` pe un Series de tip timedelta pentru a obtine zile intregi.
- Creeaza mai intai `result = df.copy()` ca sa nu modifici DataFrame-ul original.
- `reference_date` are o valoare implicita `pd.Timestamp('2024-01-01')`. Foloseste-l asa cum este; nu il recalcula din date.


In [ ]:
def create_domain_features(df, reference_date=pd.Timestamp('2024-01-01')):
    """
    Create domain-driven RFM features from customer transaction data.

    Args:
        df (pd.DataFrame): Customer DataFrame with columns:
            last_purchase_date, purchase_count, total_spent, signup_date
        reference_date (pd.Timestamp): Fixed date to compute recency from.

    Returns:
        pd.DataFrame: Copy of df with 4 new columns:
            recency_days          – integer, days since last purchase
            avg_order_value       – float, total_spent / purchase_count
            customer_lifetime_days – integer, days since signup
            purchase_frequency    – float, purchase_count / customer_lifetime_days
    """
    ### CODUL TAU AICI ###
    return None

In [ ]:
df_domain = create_domain_features(df)
print(f"Shape before: {df.shape}  |  Shape after: {df_domain.shape}")
print(f"New columns: {[c for c in df_domain.columns if c not in df.columns]}")
print(f"\nFirst 3 rows of new features:")
print(df_domain[['recency_days', 'avg_order_value', 'customer_lifetime_days', 'purchase_frequency']].head(3).to_string())

Shape before: (500, 7)  |  Shape after: (500, 11)
New columns: ['recency_days', 'avg_order_value', 'customer_lifetime_days', 'purchase_frequency']

First 3 rows of new features:
   recency_days  avg_order_value  customer_lifetime_days  purchase_frequency
0           103        58.041837                     861            0.056911
1           349       381.524167                    1260            0.009524
2           271         3.816809                     524            0.089695

In [ ]:
unittests.exercise_1(create_domain_features)

<a id='exercise-2'></a>
## Exercitiul 2 - Trasaturi matematice

### Context

Transformarile matematice permit modelelor liniare sa invete relatii neliniare si sa normalizeze efectele de scala:
- **Rapoartele** normalizeaza o cantitate prin alta cantitate inrudita (de ex. cheltuiala per produs)
- **Termenii de interactiune** (A x B) permit modelului sa raspunda la efectul combinat al doua trasaturi
- **Transformarile log** comprima distributiile asimetrice spre dreapta; `np.log1p(x)` este sigur cand x = 0
- **Termenii patratici** ii dau modelului capacitatea de a potrivi curbe

### Sarcina

Implementeaza `create_mathematical_features(df)` care adauga **4 coloane noi**:

| Coloana noua | Formula |
|:-----------|:--------|
| `spend_per_product` | `total_spent / n_products` |
| `purchase_spend_interaction` | `purchase_count x total_spent` |
| `log_total_spent` | `np.log1p(total_spent)` |
| `total_spent_squared` | `total_spent ** 2` |

**Indicii:**
- Creeaza o copie cu `result = df.copy()`.
- Foloseste `np.log1p`, nu `np.log` - gestioneaza in siguranta valorile zero.


In [ ]:
def create_mathematical_features(df):
    """
    Create mathematical transformation features.

    Args:
        df (pd.DataFrame): Customer DataFrame with columns:
            total_spent, n_products, purchase_count

    Returns:
        pd.DataFrame: Copy of df with 4 new columns:
            spend_per_product          – float, total_spent / n_products
            purchase_spend_interaction – float, purchase_count * total_spent
            log_total_spent            – float, np.log1p(total_spent)
            total_spent_squared        – float, total_spent ** 2
    """
    ### CODUL TAU AICI ###
    return None

In [ ]:
df_math = create_mathematical_features(df)
print(f"New columns: {[c for c in df_math.columns if c not in df.columns]}")
print(f"\nFirst 3 rows of new features:")
print(df_math[['spend_per_product', 'purchase_spend_interaction', 'log_total_spent', 'total_spent_squared']].head(3).to_string())

New columns: ['spend_per_product', 'purchase_spend_interaction', 'log_total_spent', 'total_spent_squared']

First 3 rows of new features:
   spend_per_product  purchase_spend_interaction  log_total_spent  total_spent_squared
0         189.603333                   139358.45         7.953336         8.088620e+06
1        1144.572500                    54939.48         8.429299         2.096074e+07
2          16.308182                     8431.33         5.195121         3.218077e+04

In [ ]:
unittests.exercise_2(create_mathematical_features)

<a id='exercise-3'></a>
## Exercitiul 3 - Trasaturi DateTime

### Context

O coloana datetime bruta este doar un numar intreg - un model liniar nu poate folosi direct "20 septembrie 2023" pentru a invata ca achizitiile din toamna difera de cele din primavara. Extragerea componentelor temporale sparge datetime-ul in semnale independente:

- **Ziua saptamanii** surprinde tipare in interiorul saptamanii (cumparatori de luni vs. cumparatori de weekend)
- **Luna / Trimestrul** surprind sezonalitatea
- **Este weekend** comprima semnalul zilei saptamanii intr-un indicator binar
- **Anul** surprinde tendinte pe termen lung

### Sarcina

Implementeaza `create_datetime_features(df)` care adauga **5 coloane noi**:

| Coloana noua | Sursa | Metoda |
|:-----------|:-------|:-------|
| `purchase_day_of_week` | `last_purchase_date` | `.dt.dayofweek` (0=Lun, 6=Dum) |
| `purchase_month` | `last_purchase_date` | `.dt.month` |
| `purchase_quarter` | `last_purchase_date` | `.dt.quarter` |
| `is_weekend_purchase` | `last_purchase_date` | 1 daca `dayofweek >= 5`, altfel 0 |
| `signup_year` | `signup_date` | `.dt.year` |

**Indicii:**
- Foloseste `.astype(int)` pentru a converti conditia booleana de weekend in 0/1.
- Toate cele cinci trasaturi provin din cele doua coloane datetime - nu este necesara aritmetica.


In [ ]:
def create_datetime_features(df):
    """
    Extract temporal features from datetime columns.

    Args:
        df (pd.DataFrame): Customer DataFrame with columns:
            last_purchase_date, signup_date

    Returns:
        pd.DataFrame: Copy of df with 5 new columns:
            purchase_day_of_week – int, 0=Monday … 6=Sunday
            purchase_month       – int, 1–12
            purchase_quarter     – int, 1–4
            is_weekend_purchase  – int, 1 if Sat/Sun else 0
            signup_year          – int
    """
    ### CODUL TAU AICI ###
    return None

In [ ]:
df_dt = create_datetime_features(df)
print(f"New columns: {[c for c in df_dt.columns if c not in df.columns]}")
print(f"\nFirst 3 rows of new features:")
print(df_dt[['purchase_day_of_week', 'purchase_month', 'purchase_quarter', 'is_weekend_purchase', 'signup_year']].head(3).to_string())
print(f"\nWeekend purchases: {df_dt['is_weekend_purchase'].sum()} out of {len(df_dt)}")

New columns: ['purchase_day_of_week', 'purchase_month', 'purchase_quarter', 'is_weekend_purchase', 'signup_year']

First 3 rows of new features:
   purchase_day_of_week  purchase_month  purchase_quarter  is_weekend_purchase  signup_year
0                     2               9                 3                    0         2021
1                     1               1                 1                    0         2020
2                     2               4                 2                    0         2022

Weekend purchases: 154 out of 500

In [ ]:
unittests.exercise_3(create_datetime_features)

<a id='exercise-4'></a>
## Exercitiul 4 - Trasaturi text

### Context

Textul brut nu poate fi introdus direct in majoritatea modelelor ML. Cea mai simpla abordare este sa extragi trasaturi numerice proxy care surprind aspecte relevante ale textului:

- **Lungimea** si **numarul de cuvinte** coreleaza cu complexitatea produsului si calitatea descrierii
- **Flag-urile de cuvinte-cheie** surprind semnale despre categoria produsului (premium, wireless) ca indicatori binari

Pentru reprezentari mai bogate ai folosi TF-IDF sau word embeddings, dar pentru multe aplicatii aceste trasaturi simple sunt surprinzator de eficiente.

### Sarcina

Implementeaza `create_text_features(df)` care adauga **4 coloane noi** din `product_name`:

| Coloana noua | Cum se calculeaza |
|:-----------|:--------------|
| `name_length` | `str.len()` |
| `word_count` | `str.split().str.len()` |
| `has_premium` | 1 daca numele contine "premium" (fara sensibilitate la litere mari/mici), altfel 0 |
| `has_wireless` | 1 daca numele contine "wireless" (fara sensibilitate la litere mari/mici), altfel 0 |

**Indicii:**
- Foloseste `str.contains('premium', case=False)` pentru cautare fara sensibilitate la litere mari/mici.
- Converteste rezultatul boolean la int cu `.astype(int)`.


In [ ]:
def create_text_features(df):
    """
    Extract numerical features from a text column.

    Args:
        df (pd.DataFrame): Customer DataFrame with column:
            product_name

    Returns:
        pd.DataFrame: Copy of df with 4 new columns:
            name_length  – int, number of characters in product_name
            word_count   – int, number of words in product_name
            has_premium  – int binary, 1 if 'premium' in name (case-insensitive) else 0
            has_wireless – int binary, 1 if 'wireless' in name (case-insensitive) else 0
    """
    ### CODUL TAU AICI ###
    return None

In [ ]:
df_text = create_text_features(df)
print(f"New columns: {[c for c in df_text.columns if c not in df.columns]}")
print(f"\nFirst 5 rows:")
print(df_text[['product_name', 'name_length', 'word_count', 'has_premium', 'has_wireless']].head(5).to_string())
print(f"\nProducts with 'premium': {df_text['has_premium'].sum()}")
print(f"Products with 'wireless': {df_text['has_wireless'].sum()}")

New columns: ['name_length', 'word_count', 'has_premium', 'has_wireless']

First 5 rows:
            product_name  name_length  word_count  has_premium  has_wireless
0        smartphone case           15           2            0             0
1  running shoes premium           21           3            1             0
2    gaming keyboard rgb           19           3            0             0
3        laptop computer           15           2            0             0
4        laptop computer           15           2            0             0

Products with 'premium': 100
Products with 'wireless': 67

In [ ]:
unittests.exercise_4(create_text_features)

### Bonus: demo TF-IDF (ne-evaluat)

Iata cum ai crea trasaturi text mai bogate folosind TF-IDF. Aceasta celula este pentru explorare - nu este testata.


In [ ]:
vectorizer = TfidfVectorizer(max_features=10)
tfidf_matrix = vectorizer.fit_transform(df['product_name'])
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=vectorizer.get_feature_names_out()
)
print("TF-IDF feature matrix shape:", tfidf_df.shape)
print("\nFeature names:", list(tfidf_df.columns))
print("\nFirst 3 rows:")
print(tfidf_df.head(3).round(4).to_string())

<a id='exercise-5'></a>
## Exercitiul 5 - Trasaturi polinomiale (scikit-learn)

### Context

`PolynomialFeatures` din scikit-learn genereaza sistematic toti termenii polinomiali si de interactiune pana la un anumit grad. Pentru 3 trasaturi de intrare [x1, x2, x3] la gradul 2:

$$[x_1, x_2, x_3] \rightarrow [x_1, x_2, x_3, x_1^2, x_1 x_2, x_1 x_3, x_2^2, x_2 x_3, x_3^2]$$

Acest lucru ofera modelului capacitatea de a invata automat relatii neliniare si interactiuni intre trasaturi.

**Atentie:** numarul de trasaturi creste combinatorial. Aplica intotdeauna regularizare sau selectie de trasaturi dupa folosirea `PolynomialFeatures`.

### Sarcina

Implementeaza `create_polynomial_features(X, degree=2)` folosind `sklearn.preprocessing.PolynomialFeatures`:
- seteaza `include_bias=False` (nu vrem un termen constant)
- fa fit si transforma `X` intr-un singur pas cu `.fit_transform(X)`

**Indicii:**
- Functia primeste un array NumPy `X`, nu un DataFrame.
- Pentru 3 trasaturi de intrare si degree=2, iesirea are 9 coloane.
- Pentru 3 trasaturi de intrare si degree=3, iesirea are 19 coloane.


In [ ]:
def create_polynomial_features(X, degree=2):
    """
    Generate polynomial and interaction features using scikit-learn.

    Args:
        X (np.ndarray): Input feature matrix, shape (n_samples, n_features)
        degree (int): Maximum polynomial degree.

    Returns:
        np.ndarray: Transformed feature matrix.
            For 3 input features and degree=2, output shape is (n_samples, 9).
    """
    ### CODUL TAU AICI ###
    return None

In [ ]:
X_poly = create_polynomial_features(X_num, degree=2)
print(f"Input shape:          {X_num.shape}")
print(f"Output shape (deg=2): {X_poly.shape}")
print(f"Output shape (deg=3): {create_polynomial_features(X_num, degree=3).shape}")
print(f"\nFirst row — original: {X_num[0]}")
print(f"First row — poly:     {X_poly[0]}")

Input shape:          (500, 3)
Output shape (deg=2): (500, 9)
Output shape (deg=3): (500, 19)

First row — original: [  49.   2844.05   15.  ]
First row — poly:     [4.90000000e+01 2.84405000e+03 1.50000000e+01 2.40100000e+03
 1.39358450e+05 7.35000000e+02 8.08862002e+06 4.26607500e+04
 2.25000000e+02]

In [ ]:
unittests.exercise_5(create_polynomial_features)

<a id='exercise-6'></a>
## Exercitiul 6 - Trasaturi polinomiale de la zero

### Context

Acum implementeaza aceeasi transformare fara scikit-learn.

Ideea-cheie este ca fiecare termen polinomial de grad d este produsul a d indici de coloane din intrare. Functia `itertools.combinations_with_replacement(range(n_features), d)` genereaza toate aceste tuple de indici in ordine sortata - exact ca iesirea produsa de sklearn.

**Algoritm:**
1. Porneste cu `feature_cols = [X]` (trasaturile originale)
2. Pentru fiecare grad d de la 2 la `degree`:
   - Pentru fiecare combinatie de d indici de coloane (cu repetitie permisa):
     - Calculeaza noua coloana ca `np.prod([X[:, i] for i in combo], axis=0)`
     - Redimensioneaz-o la `(-1, 1)` si adaug-o in `feature_cols`
3. Returneaza `np.hstack(feature_cols)`

### Sarcina

Implementeaza `polynomial_features_from_scratch(X, degree=2)`.

**Indicii:**
- `combinations_with_replacement(range(3), 2)` returneaza: `(0,0), (0,1), (0,2), (1,1), (1,2), (2,2)` - exact cei 6 termeni de grad 2.
- `np.prod([X[:, 0], X[:, 1]], axis=0)` calculeaza produsul element-cu-element al doua coloane.
- Iesirea ta trebuie sa se potriveasca exact cu `sklearn.preprocessing.PolynomialFeatures(degree=..., include_bias=False).fit_transform(X)`.


In [ ]:
def polynomial_features_from_scratch(X, degree=2):
    """
    Generate polynomial and interaction features WITHOUT using sklearn.

    Use itertools.combinations_with_replacement to enumerate all degree-d
    feature combinations and np.prod to compute each new term.

    Args:
        X (np.ndarray): Input feature matrix, shape (n_samples, n_features)
        degree (int): Maximum polynomial degree.

    Returns:
        np.ndarray: Transformed feature matrix matching sklearn's output exactly.
    """
    ### CODUL TAU AICI ###
    return None

In [ ]:
X_scratch = polynomial_features_from_scratch(X_num, degree=2)
X_sklearn = create_polynomial_features(X_num, degree=2)

print(f"Shape (from scratch):  {X_scratch.shape}")
print(f"Shape (sklearn):       {X_sklearn.shape}")
print(f"Matches sklearn:       {np.allclose(X_scratch, X_sklearn)}")
print(f"\nFirst row — from scratch: {X_scratch[0]}")

Shape (from scratch):  (500, 9)
Shape (sklearn):       (500, 9)
Matches sklearn:       True

First row — from scratch: [4.90000000e+01 2.84405000e+03 1.50000000e+01 2.40100000e+03
 1.39358450e+05 7.35000000e+02 8.08862002e+06 4.26607500e+04
 2.25000000e+02]

In [ ]:
unittests.exercise_6(polynomial_features_from_scratch)

## Rezumat

Ai construit sase seturi de trasaturi pornind din date brute despre tranzactiile clientilor:

| Exercitiu | Trasaturi create | Tehnica |
|:---------|:----------------|:----------|
| 1 | `recency_days`, `avg_order_value`, `customer_lifetime_days`, `purchase_frequency` | Cunostinte de domeniu (RFM) |
| 2 | `spend_per_product`, `purchase_spend_interaction`, `log_total_spent`, `total_spent_squared` | Transformari matematice |
| 3 | `purchase_day_of_week`, `purchase_month`, `purchase_quarter`, `is_weekend_purchase`, `signup_year` | Extragere DateTime |
| 4 | `name_length`, `word_count`, `has_premium`, `has_wireless` | Trasaturi text |
| 5 | Matrice polinomiala cu 9 coloane | `sklearn.PolynomialFeatures` |
| 6 | Matrice polinomiala cu 9 coloane | `combinations_with_replacement` |

**Idei-cheie:**
- Trasaturile bune bat aproape intotdeauna modelele complexe.
- Foloseste intotdeauna o data de referinta fixa pentru trasaturile bazate pe timp.
- Transformarile log trebuie aplicate inainte de scalare.
- Trasaturile polinomiale explodeaza ca dimensiune - foloseste regularizare.
